# Module X4: Time-series analysis and inference

**Goal:** analyse data that come **in time order**, where today's value depends on yesterday's. You will split a series into **trend, seasonality and remainder** by hand, measure **autocorrelation**, see exactly **why it breaks the independence assumption** behind the tests of Modules 8 and 8b, repair the standard errors (**HAC**) or the model (**AR errors**), avoid **spurious regressions** by differencing, and **forecast honestly** against simple baselines.

**You will learn:** decomposition (moving averages, seasonal indices) · autocorrelation (ACF, PACF) · effective sample size · Durbin–Watson · HAC (Newey–West) standard errors · stationarity and the ADF test · differencing · spurious regression · AR(1) models, forecasts and forecast uncertainty · regression with AR errors (Cochrane–Orcutt) · chronological train/test splits · MAE, RMSE and MASE

**Maths in this module:** sample autocovariance and autocorrelation · $n_\text{eff}=n\frac{1-\phi}{1+\phi}$ · AR(1) algebra ($\mu+\phi^h(y_T-\mu)$ and its variance) · least squares for AR(1) · scaled forecast errors

> 🔗 **Prerequisites:** Modules 8 and 8b (regression, standard errors, p-values) and 7 (variance). Needs `statsmodels` (already used since Module 9).

In [ ]:
# Setup: find the course data folder (works locally and in Google Colab)
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    import subprocess
    if not Path("/content/data-analysis-learnbook").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/PathornKiti/data-analysis-learnbook.git",
                        "/content/data-analysis-learnbook"], check=True)
    DATA = Path("/content/data-analysis-learnbook/data")
else:
    DATA = next(p / "data" for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "data" / "students.csv").exists())
print("Data folder:", DATA)

In [ ]:
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.stattools import durbin_watson
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import acf, adfuller, pacf

sns.set_theme(style="whitegrid", palette="colorblind")

dem = pd.read_csv(DATA / "monthly_demand.csv", parse_dates=["month"])
n = len(dem)
y = dem["demand"].to_numpy()
dem["t"] = np.arange(n)            # month number 0..143
dem["years"] = dem["t"] / 12       # the same clock, in years
dem["mon"] = dem["month"].dt.month
print(dem.head(3).to_string(index=False))
print(f"{n} months, {dem['month'].min():%b %Y} to {dem['month'].max():%b %Y}; promotion in {int(dem['promo'].sum())} of them")

## 1. What makes a time series different

`monthly_demand.csv` holds twelve years of monthly demand for a product (synthetic), plus a flag for months with a **promotion**. Three things are visible at a glance.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.8))
ax.plot(dem["month"], dem["demand"], color="grey", lw=1)
promo = dem[dem["promo"] == 1]
ax.scatter(promo["month"], promo["demand"], s=22, color="C1", zorder=3, label="promotion month")
ax.set(xlabel="month", ylabel="demand", title="Monthly demand: trend, yearly cycle, promotions")
ax.legend()
plt.show()

* a **trend** (demand rises over time),
* **seasonality** (a repeating yearly shape),
* and a **remainder** that is not pure noise: a high month tends to be followed by another high month.

That last point is the heart of this module. Almost every method you have used so far assumed that observations are **independent**. In a time series that is usually false, and the cost is *standard errors that are too small, p-values that are too small, and "significant" results that are not*.

An **additive** model writes

$$y_t=\underbrace{T_t}_{\text{trend}}+\underbrace{S_t}_{\text{season}}+\underbrace{R_t}_{\text{remainder}},\qquad S_{t+12}=S_t .$$

(If the seasonal swings grow with the level, a **multiplicative** model $y_t=T_tS_tR_t$ is better; take logs to make it additive.)

## 2. Decomposition by hand

**Trend:** average over exactly one full year so the seasonal pattern cancels. With monthly data the window of 12 has no centre month, so we use a **centred $2\times12$ moving average**: weights $\tfrac1{12}\cdot(\tfrac12,1,1,\dots,1,\tfrac12)$ (13 weights). The first and last 6 months have no trend value.

**Seasonal indices:** subtract the trend, average the leftover by calendar month, and shift the 12 averages to sum to zero. **Remainder:** what is left.

In [ ]:
w = np.r_[0.5, np.ones(11), 0.5] / 12
trend = np.full(n, np.nan)
trend[6:n - 6] = np.convolve(y, w, mode="valid")

detrended = y - trend
month_avg = np.array([np.nanmean(detrended[m::12]) for m in range(12)])   # series starts in January
season_idx = month_avg - month_avg.mean()
seasonal = np.tile(season_idx, n // 12)
remainder = y - trend - seasonal

print("seasonal indices Jan..Dec:", np.round(season_idx, 1))
print("sum of indices:", round(season_idx.sum(), 10))
print("strongest month:", int(season_idx.argmax()) + 1, " weakest month:", int(season_idx.argmin()) + 1)
print(f"remainder: {np.isfinite(remainder).sum()} valid months, sd = {np.nanstd(remainder, ddof=1):.2f}")

ref = seasonal_decompose(y, model="additive", period=12)
print("matches statsmodels (trend, seasonal, remainder):",
      np.allclose(trend, ref.trend, equal_nan=True),
      np.allclose(seasonal, ref.seasonal),
      np.allclose(remainder, ref.resid, equal_nan=True))

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(10, 8), sharex=True)
for ax, (name, s) in zip(axes, [("observed", y), ("trend", trend), ("seasonal", seasonal), ("remainder", remainder)]):
    ax.plot(dem["month"], s, lw=1, color="C0" if name != "remainder" else "C2")
    ax.set_ylabel(name)
axes[-1].axhline(0, color="k", lw=0.5)
plt.show()

Exactly what a library does, now with no mystery: a moving average, 12 monthly means and a subtraction. The data were built as $200+0.8t+12\sin(\ldots)$ + promotion + autocorrelated noise, so the demand peaks around **June** (month 6) and bottoms out in late autumn (**November/December**), as the indices show.

Notice what decomposition did **not** do: it ignored the promotion flag and treats everything not trend or season as "remainder". The remainder therefore contains the promotion effect *and* the autocorrelated noise.

## 3. Autocorrelation

The **autocorrelation at lag $k$** is the correlation of a series with itself $k$ steps earlier:

$$r_k=\frac{\sum_{t=k+1}^{n}(x_t-\bar x)(x_{t-k}-\bar x)}{\sum_{t=1}^{n}(x_t-\bar x)^2}.$$

(The denominator is the *full* sum of squares, which is the convention in statsmodels' `acf`.) If a series were independent, each $r_k$ would be within about $\pm1.96/\sqrt n$ of zero 95% of the time; these are the dashed bands on an **ACF plot**.

In [ ]:
def acf_by_hand(x, k):
    x = np.asarray(x, dtype=float)
    d = x - x.mean()
    return (d[k:] * d[:len(d) - k]).sum() / (d ** 2).sum()

rem = remainder[np.isfinite(remainder)]
print("raw demand       lags 1, 2, 12:", [round(acf_by_hand(y, k), 3) for k in (1, 2, 12)])
print("remainder        lags 1, 2, 3 :", [round(acf_by_hand(rem, k), 3) for k in (1, 2, 3)])
print("matches statsmodels acf:", np.allclose([acf_by_hand(y, k) for k in range(1, 13)], acf(y, nlags=12)[1:]))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4), sharey=True)
for ax, (name, s) in zip(axes, [("raw demand", y), ("remainder", rem), ("remainder PACF", rem)]):
    k = np.arange(1, 25)
    if name.endswith("PACF"):
        vals = pacf(s, nlags=24)[1:]
    else:
        vals = acf(s, nlags=24)[1:]
    ax.bar(k, vals, color="C0", width=0.6)
    band = 1.96 / math.sqrt(len(s))
    ax.axhline(band, ls="--", color="grey"); ax.axhline(-band, ls="--", color="grey"); ax.axhline(0, color="k", lw=0.5)
    ax.set(title=name, xlabel="lag (months)")
axes[0].set_ylabel("correlation")
plt.show()

* The **raw** series has huge, slowly-decaying autocorrelation (0.95 at lag 1): that is mostly the *trend* talking.
* The **remainder** has far less, but lag 1 (0.36) is clearly outside the bands: real short-term dependence is left.
* The **PACF** ("the correlation at lag $k$ after removing lags $1\ldots k-1$") is the usual tool to pick the order of an AR model; a single spike at lag 1 suggests AR(1).

One caution you will meet again later: the remainder understates the true dependence. The data were generated with an AR(1) coefficient of **0.6**, yet the remainder shows 0.36, because the moving-average trend soaks up slow wandering and the promotion effect adds a bit of noise. Decomposition does not fully isolate "the noise".

## 4. Why autocorrelation breaks the usual tests

For $n$ independent observations the mean has variance $\sigma^2/n$. For an AR(1) series with correlation $\phi$ between neighbours, the variance of the mean is, for large $n$, about $\frac{\sigma^2}{n}\cdot\frac{1+\phi}{1-\phi}$. It is as if you had only

$$n_\text{eff}=n\,\frac{1-\phi}{1+\phi}$$

independent observations. With $n=144$ and $\phi=0.6$ that is **36**: one quarter of the nominal sample size, and standard errors **twice** as large as the formula for independent data would say. Check by simulation:

In [ ]:
def ar1_series(n, phi, rng, sd=1.0, burn=100):
    e = rng.normal(0, sd, n + burn)
    x = np.zeros(n + burn)
    for i in range(1, n + burn):
        x[i] = phi * x[i - 1] + e[i]
    return x[burn:]

rng = np.random.default_rng(1)
phi, nn, reps = 0.6, 144, 5000
means, naive_se = [], []
for _ in range(reps):
    s = ar1_series(nn, phi, rng)
    means.append(s.mean())
    naive_se.append(s.std(ddof=1) / math.sqrt(nn))
marginal_sd = 1 / math.sqrt(1 - phi ** 2)

print(f"typical 'iid' SE of the mean            : {np.mean(naive_se):.4f}")
print(f"actual sd of the mean over {reps} series : {np.std(means):.4f}")
print(f"formula, marginal sd * sqrt((1+phi)/((1-phi) n)): {marginal_sd * math.sqrt((1 + phi) / ((1 - phi) * nn)):.4f}")
print(f"n_eff = {nn * (1 - phi) / (1 + phi):.0f}")

The usual standard error is about **half** of the real variability of the mean. Confidence intervals built from it are too narrow, and p-values too small.

### Detecting it: the Durbin–Watson statistic

For regression residuals $e_t$,

$$DW=\frac{\sum_{t=2}^{n}(e_t-e_{t-1})^2}{\sum_{t=1}^{n}e_t^2}\approx 2(1-r_1).$$

$DW\approx2$ means no lag-1 autocorrelation; $DW\to0$ strong positive; $DW\to4$ strong negative. Let us fit the natural regression: demand on **time, promotion and month dummies**.

In [ ]:
X = pd.get_dummies(dem["mon"], prefix="m", drop_first=True).astype(float)
X.insert(0, "years", dem["years"])
X.insert(1, "promo", dem["promo"].astype(float))
X.insert(0, "const", 1.0)
Xn = X.to_numpy()
names = list(X.columns)

ols = sm.OLS(y, Xn).fit()
e = ols.resid
dw_hand = (np.diff(e) ** 2).sum() / (e ** 2).sum()
r1 = (e[1:] * e[:-1]).sum() / (e ** 2).sum()
print(f"Durbin-Watson by hand {dw_hand:.3f}  (statsmodels {durbin_watson(e):.3f});  2(1 - r1) = {2 * (1 - r1):.3f}")
print(f"residual lag-1 autocorrelation: {r1:.3f}")
print(f"trend: {ols.params[1]:.2f} per year;  promotion effect: {ols.params[2]:.2f}")

$DW\approx0.78$, far below 2: residuals are strongly positively autocorrelated, so the OLS standard errors cannot be trusted. (The OLS *coefficients* are still unbiased here; it is the *uncertainty* that is wrong.)

### Fix 1: HAC (Newey–West) standard errors

Keep the OLS coefficients but replace the covariance with one that allows the errors to be autocorrelated up to `maxlags` months ("heteroskedasticity- and autocorrelation-consistent"). In statsmodels this is one argument.

In [ ]:
hac = sm.OLS(y, Xn).fit(cov_type="HAC", cov_kwds={"maxlags": 6})
i_promo, i_years = names.index("promo"), names.index("years")
rows = []
for label, i in (("promo", i_promo), ("years (trend)", i_years)):
    rows.append({"term": label, "estimate": ols.params[i], "naive SE": ols.bse[i], "HAC SE": hac.bse[i],
                 "HAC / naive": hac.bse[i] / ols.bse[i]})
print(pd.DataFrame(rows).round(3).to_string(index=False))

Two different stories, and both are instructive:

* For the **trend** (a slowly changing regressor), the HAC standard error is about **46% larger**: the naive one was badly over-confident.
* For the **promotion** (assigned essentially at random, so unrelated to the slowly drifting noise) the HAC standard error is slightly **smaller** than the naive one. Autocorrelation hurts most when the *regressor* is itself persistent. Do not assume a robust standard error is always the bigger one.

### How bad can it get? Simulation

Take two series that are **independent by construction** and test whether one "predicts" the other at the 5% level. A valid test should reject about 5% of the time.

In [ ]:
def false_positive_rates(make_pair, reps=1000, seed=0, maxlags=4):
    rng = np.random.default_rng(seed)
    naive = robust = 0
    for _ in range(reps):
        a, b = make_pair(rng)
        Xs = sm.add_constant(a)
        naive += sm.OLS(b, Xs).fit().pvalues[1] < 0.05
        robust += sm.OLS(b, Xs).fit(cov_type="HAC", cov_kwds={"maxlags": maxlags}).pvalues[1] < 0.05
    return naive / reps, robust / reps

scenarios = {
    "independent noise": lambda r: (r.normal(size=100), r.normal(size=100)),
    "two independent AR(1), phi = 0.8": lambda r: (ar1_series(100, 0.8, r), ar1_series(100, 0.8, r)),
    "two independent random walks": lambda r: (np.cumsum(r.normal(size=100)), np.cumsum(r.normal(size=100))),
    "two independent random walks, differenced": lambda r: (np.diff(np.cumsum(r.normal(size=101))), np.diff(np.cumsum(r.normal(size=101)))),
}
out = []
for name, f in scenarios.items():
    a, b = false_positive_rates(f)
    out.append({"scenario": name, "naive rejects": a, "HAC rejects": b})
print(pd.DataFrame(out).to_string(index=False))

Nominal 5%, but:

* independent AR(1) series with $\phi=0.8$: the naive test "finds" a relationship **about a third of the time** (35%); HAC cuts that to about 21%, still far above 5% with 100 points and persistent series (HAC works best with long series and moderate dependence);
* independent **random walks**: the naive test is wrong about three times out of four (74%), and even HAC is wrong about 60% of the time. This is the famous **spurious regression**: two series that both wander will look related.
* the last row shows the cure for random walks: **difference** them first. The differences of a random walk are independent noise, so the naive test is valid again (6%, within simulation error of 5%). Note that HAC is slightly liberal in small samples (it rejects 7% even for pure noise, 9% for the differenced walks), which is one reason to prefer fixing the model when you can.

## 5. Stationarity and differencing

A series is (weakly) **stationary** if its mean, variance and autocorrelations do not change over time. Trends and random walks are not. Most time-series methods need stationarity, which you usually get by **differencing** ($\Delta y_t=y_t-y_{t-1}$) or by removing trend and season.

The **augmented Dickey–Fuller (ADF) test** has the null hypothesis *"the series has a unit root (a random-walk-like, non-stationary behaviour)"*. A **small** p-value is evidence of stationarity. Careful with the direction: here you *want* to reject.

In [ ]:
def adf(s):
    with warnings.catch_warnings():              # silence a statsmodels notice about a future return type
        warnings.simplefilter("ignore", FutureWarning)
        stat, p = adfuller(s)[:2]
    return stat, p

for name, s in (("demand", y), ("first difference of demand", np.diff(y)), ("decomposition remainder", rem)):
    stat, p = adf(s)
    print(f"{name:28s} ADF statistic {stat:7.2f}   p = {p:.4f}")

The raw series cannot reject a unit root (p ≈ 0.94, because of the trend). The first difference and the remainder are comfortably stationary (p < 0.0001). Two caveats: the ADF test has low power against slowly-mean-reverting series, and a trend-stationary series (trend + stationary noise, as here) is better handled by *modelling the trend* than by differencing.

A random walk drawn by simulation shows the failure mode directly:

In [ ]:
rng = np.random.default_rng(7)
rw = np.cumsum(rng.normal(size=300))
print(f"random walk:            ADF p = {adf(rw)[1]:.3f}")
print(f"its first difference:   ADF p = {adf(np.diff(rw))[1]:.4f}")

## 6. A model for the dependence: AR(1)

An **autoregressive model of order 1** says each value is a pull back toward a mean, plus fresh noise:

$$x_t-\mu=\phi\,(x_{t-1}-\mu)+\varepsilon_t,\qquad \varepsilon_t\sim(0,\sigma^2),\ |\phi|<1 .$$

**Estimating it is just regression**: regress $x_t$ on $x_{t-1}$. Use the decomposition remainder as the series:

In [ ]:
def fit_ar1(x):
    x = np.asarray(x, dtype=float)
    A = np.column_stack([np.ones(len(x) - 1), x[:-1]])
    coef, *_ = np.linalg.lstsq(A, x[1:], rcond=None)
    c, phi_hat = coef
    resid = x[1:] - A @ coef
    sigma = math.sqrt((resid ** 2).sum() / (len(resid) - 2))
    return c, phi_hat, sigma

c_hat, phi_hat, sigma_hat = fit_ar1(rem)
mu_hat = c_hat / (1 - phi_hat)
print(f"phi = {phi_hat:.3f}, sigma = {sigma_hat:.2f}, mean mu = {mu_hat:.2f}")
lib = AutoReg(rem, lags=1, trend="c").fit()
print("matches statsmodels AutoReg:", np.allclose([c_hat, phi_hat], lib.params))

### Forecasting an AR(1)

Iterating the model $h$ steps forward from the last observation $x_T$:

$$\hat x_{T+h}=\mu+\phi^{h}(x_T-\mu),\qquad \operatorname{sd}(x_{T+h}-\hat x_{T+h})=\sigma\sqrt{\frac{1-\phi^{2h}}{1-\phi^{2}}} .$$

The forecast **decays geometrically to the mean** and the uncertainty **grows to the stationary sd** $\sigma/\sqrt{1-\phi^2}$. (This ignores that $\mu,\phi,\sigma$ are themselves estimated: real intervals are a bit wider.) A simulation confirms the variance formula:

In [ ]:
def ar1_forecast(mu, phi, sigma, last, h):
    mean = mu + phi ** h * (last - mu)
    sd = sigma * math.sqrt((1 - phi ** (2 * h)) / (1 - phi ** 2))
    return mean, sd

mu0, phi0, sig0, last = 100.0, 0.6, 4.0, 115.0
rng = np.random.default_rng(3)
paths = np.empty((20000, 6))
cur = np.full(20000, last)
for h in range(6):
    cur = mu0 + phi0 * (cur - mu0) + rng.normal(0, sig0, 20000)
    paths[:, h] = cur
print(" h   formula mean  sim mean   formula sd  sim sd")
for h in (1, 2, 4, 6):
    m, s = ar1_forecast(mu0, phi0, sig0, last, h)
    print(f"{h:2d}   {m:10.2f} {paths[:, h - 1].mean():9.2f} {s:11.2f} {paths[:, h - 1].std():8.2f}")
print(f"stationary sd = {sig0 / math.sqrt(1 - phi0 ** 2):.2f}")

## 7. Regression *with* AR errors (Cochrane–Orcutt)

HAC repaired the standard errors but left the coefficients as they were. If the errors follow $u_t=\rho\,u_{t-1}+\varepsilon_t$, a more efficient fix is to **quasi-difference**: subtract $\rho$ times the previous row from every row, which turns $u_t$ into the independent $\varepsilon_t$:

$$y_t-\rho y_{t-1}=\beta^\top(x_t-\rho x_{t-1})+\varepsilon_t .$$

$\rho$ is unknown, so alternate: fit OLS → estimate $\rho$ from the residuals → quasi-difference and refit → repeat until it settles.

In [ ]:
def cochrane_orcutt(y, X, n_iter=50):
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    for _ in range(n_iter):
        u = y - X @ beta
        rho = (u[1:] * u[:-1]).sum() / (u[:-1] ** 2).sum()
        ys, Xs = y[1:] - rho * y[:-1], X[1:] - rho * X[:-1]
        new = np.linalg.lstsq(Xs, ys, rcond=None)[0]
        done = np.allclose(new, beta, atol=1e-10)
        beta = new
        if done:
            break
    resid = ys - Xs @ beta
    s2 = (resid ** 2).sum() / (len(ys) - X.shape[1])
    se = np.sqrt(np.diag(s2 * np.linalg.inv(Xs.T @ Xs)))
    return beta, se, rho

beta_co, se_co, rho_co = cochrane_orcutt(y, Xn)
glsar = sm.GLSAR(y, Xn, rho=1)
res = glsar.iterative_fit(maxiter=20)
print(f"rho by hand {rho_co:.4f}, statsmodels GLSAR {float(glsar.rho[0]):.4f}")
print(f"largest difference in coefficients {np.abs(beta_co - res.params).max():.4f}, in standard errors {np.abs(se_co - res.bse).max():.4f}")

# (statsmodels estimates rho by Yule-Walker, a slightly different formula, so agreement is close but not exact)
rows = []
for label, i in (("promo", i_promo), ("years (trend)", i_years)):
    rows.append({"term": label, "OLS": ols.params[i], "OLS SE": ols.bse[i], "HAC SE": hac.bse[i],
                 "AR-errors est.": beta_co[i], "AR-errors SE": se_co[i]})
print(pd.DataFrame(rows).round(3).to_string(index=False))
lo, hi = beta_co[i_promo] - 1.96 * se_co[i_promo], beta_co[i_promo] + 1.96 * se_co[i_promo]
print(f"promotion effect, AR-errors 95% interval: ({lo:.2f}, {hi:.2f});  naive OLS interval: "
      f"({ols.params[i_promo] - 1.96 * ols.bse[i_promo]:.2f}, {ols.params[i_promo] + 1.96 * ols.bse[i_promo]:.2f})")

The promotion effect moves from 9.1 (naive OLS) to **7.5**, and its standard error shrinks from 1.44 to 0.94: removing the autocorrelated noise makes the estimate both *different* and *more precise*, since the promotion is unrelated to that noise. The data were generated with a true effect of **8.0** and a true trend of 9.6 per year; we can only know that because they are synthetic, and both methods' intervals contain 8.0 here. With real data you would not have this answer key; this is why simulation checks (Section 4) matter.

## 8. Forecasting honestly

A model is only as good as its predictions on data **it has not seen**. Rules for time series:

1. **Split chronologically.** Train on the past, test on the future. A random split lets the model "peek" at neighbours of the test points.
2. **Always beat a baseline.** The **naive** forecast repeats the last value; the **seasonal naive** repeats the value from a year earlier. A fancy model that cannot beat them is not useful.
3. **Score with error measures.** $\text{MAE}=\overline{|e|}$, $\text{RMSE}=\sqrt{\overline{e^2}}$ (punishes big misses), and **MASE** = MAE divided by the MAE of the one-year-seasonal-naive forecast *on the training data*, so it has no units: below 1 means better than that benchmark in-sample.

Train on the first 120 months, test on the last 24.

In [ ]:
tr, h = 120, 24
ytr, yte = y[:tr], y[tr:]
scale = np.mean(np.abs(ytr[12:] - ytr[:-12]))

pred = {}
pred["naive (last value)"] = np.repeat(ytr[-1], h)
pred["seasonal naive"] = np.array([ytr[tr - 12 + (k % 12)] for k in range(h)])

b_ols = np.linalg.lstsq(Xn[:tr], ytr, rcond=None)[0]
pred["regression (OLS)"] = Xn[tr:] @ b_ols

b_co, se_tr, rho_tr = cochrane_orcutt(ytr, Xn[:tr])
u_last = ytr[-1] - Xn[tr - 1] @ b_co
pred["regression + AR(1) errors"] = np.array([Xn[tr + k] @ b_co + rho_tr ** (k + 1) * u_last for k in range(h)])

def scores(yt, yp):
    e = yt - yp
    mae = np.mean(np.abs(e))
    return {"MAE": mae, "RMSE": math.sqrt(np.mean(e ** 2)), "MASE": mae / scale}

table = pd.DataFrame({k: scores(yte, v) for k, v in pred.items()}).T
print(f"MASE scale (training seasonal-naive MAE): {scale:.2f}")
print(table.round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.8))
ax.plot(dem["month"][:tr], ytr, color="grey", lw=1, label="train")
ax.plot(dem["month"][tr:], yte, color="k", lw=1.5, label="test (actual)")
for k, c in (("naive (last value)", "C3"), ("seasonal naive", "C1"), ("regression + AR(1) errors", "C0")):
    ax.plot(dem["month"][tr:], pred[k], color=c, lw=1.3, label=k)
ax.legend(ncol=2, fontsize=8)
ax.set(xlabel="month", ylabel="demand", title="Forecasts for the last 24 months")
plt.show()

* Both baselines are mediocre (MAE ≈ 10-11) because they ignore the **trend** (naive) or the **recent level** (seasonal naive lags a year of growth).
* The regression models halve the error (MAE ≈ 5-6; MASE ≈ 0.4-0.5).
* Plain OLS (MAE 5.1) is slightly *better* than OLS with AR errors (5.5) on this split. That is not a contradiction: AR errors improve the **standard errors and the efficiency of the coefficients**, but over a 24-month horizon the AR correction $\rho^h u_T$ has faded to nothing after a few steps, so it adds little to point forecasts. With only 24 test points a difference of 0.5 is within noise. Report the table, do not crown a winner.
* The honest ranking that *is* robust: **both regression models beat both baselines by a wide margin.**

**Rolling-origin evaluation** (train up to month 96, forecast 12, then train up to 108, forecast 12, and so on) gives several test periods and a better picture of stability. It is the time-series version of cross-validation, and it is always chronological.

## 9. Pitfalls and reporting

* Treating autocorrelated observations as independent: **p-values too small**, intervals too narrow.
* Regressing one trending series on another and calling the relationship real (**spurious regression**).
* Random or shuffled train/test splits (**leakage** from the future).
* Reading a decomposition's remainder as "the noise" (it mixes noise, other effects and some absorbed dependence).
* Using the ADF test the wrong way round (rejecting means *stationary*).
* Giving point forecasts without a horizon and an error measure; intervals that ignore the growth of uncertainty with $h$.
* A model that cannot beat the **naive** and **seasonal naive** baselines.

**Reporting template.**
> Monthly demand (n = 144, Jan 2013 to Dec 2024) was modelled by OLS on a linear trend, calendar-month dummies and a promotion flag. Residuals were strongly autocorrelated (Durbin–Watson 0.78; lag-1 autocorrelation 0.61), so standard errors were computed by Cochrane–Orcutt AR(1) errors (ρ = 0.63), with HAC (6 lags) as a robustness check. Promotions were associated with +7.5 units (95% CI 5.65 to 9.34) after adjusting for trend and season. On a chronological hold-out (months 121-144) the model reached MAE 5.5 (MASE 0.46) against 9.9 for the naive and 10.8 for the seasonal-naive forecasts. Residuals were not diagnosed beyond lag 1; the series is synthetic.

**Where next:** ARIMA/SARIMA and exponential smoothing (`statsmodels.tsa`), Prophet-style trend/season models, vector autoregressions (several series), cointegration (when related random walks can be modelled together), state-space models, and proper probabilistic forecast scoring.